# How to create the vector database

created using an adapted version of this: https://docs.pinecone.io/guides/search/hybrid-search#use-a-single-hybrid-index

In [ ]:
import json
import os
from pinecone import Pinecone
from pinecone import ServerlessSpec
from sentence_transformers import SentenceTransformer
from pinecone_text.sparse import BM25Encoder
from typing import List

# Initialize Pinecone
print("Initializing Pinecone...")
PINECONE_API_KEY = os.environ.get('PINECONE_API_KEY')
INDEX_NAME = 'help-centre-hybrid'
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

# load models
print("Loading models...")
dense_model = SentenceTransformer('all-MiniLM-L6-v2')
sparse_model = BM25Encoder().default()

# Create index if it doesn't exist (with hybrid search enabled)
print("Creating index...")
if INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=INDEX_NAME,
        vector_type="dense",
        dimension=dense_model.get_sentence_embedding_dimension(),
        metric='dotproduct',
        spec=ServerlessSpec(
            cloud='aws',
            region='us-east-1'
        )
    )
# Connect to the index
index = pc.Index(INDEX_NAME)

# load data
print("Loading data...")
with open('../data/chunks.json', 'r') as f:
    knowledge_base = json.load(f)

# create vectors
print("Creating vectors...")
# sparse_model.fit(knowledge_base)

# # good practice to store the values to a json file
# # store the values to a json file
# sparse_model.dump("bm25_values.json")

# load to your BM25Encoder object
sparse_model = BM25Encoder().load("../data/bm25_values.json")

/Users/adambourne/GitHub/help-centre-assistant/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Initializing Pinecone...
Loading models...
Creating index...
Loading data...
Creating vectors...


In [2]:
# Create vectors and upsert to Pinecone
for i, chunk in enumerate(knowledge_base):
    # Create dense vector embedding
    dense_vector = dense_model.encode(chunk).tolist()
    
    # Create sparse vector using BM25
    sparse_vector = sparse_model.encode_documents(chunk)
    
    # Convert sparse vector to Pinecone format
    sparse_values = sparse_vector["values"]
    sparse_indices = sparse_vector["indices"]
    
    # Upsert to Pinecone with both dense and sparse vectors
    index.upsert(
        vectors=[
            {
                "id": str(i),
                "values": dense_vector,
                "sparse_values": {
                    "values": sparse_values,
                    "indices": sparse_indices
                },
                "metadata": {
                    "id": str(i),
                    "text": chunk
                }
            }
        ]
    )

print(f"Successfully uploaded {len(knowledge_base)} chunks to Pinecone index '{INDEX_NAME}' with hybrid search enabled")

Successfully uploaded 21 chunks to Pinecone index 'help-centre-hybrid' with hybrid search enabled


In [3]:
def hybrid_score_norm(dense, sparse, alpha: float):
    """Hybrid score using a convex combination

    alpha * dense + (1 - alpha) * sparse

    Args:
        dense: Array of floats representing
        sparse: a dict of `indices` and `values`
        alpha: scale between 0 and 1
    """
    if alpha < 0 or alpha > 1:
        raise ValueError("Alpha must be between 0 and 1")
    hs = {
        'indices': sparse['indices'],
        'values':  [v * (1 - alpha) for v in sparse['values']]
    }
    return [v * alpha for v in dense], hs


def hybrid_search(query: str, top_k: int = 5, alpha: float = 0.5):
    """
    Perform hybrid search using both dense and sparse vectors
    alpha: 0.0 = sparse (BM25) only, 1.0 = dense only, 0.5 = equal weight
    """
    if not 0 <= alpha <= 1:
        raise ValueError("Alpha must be between 0 and 1")
        
    # Get dense vector for query
    dense_query = dense_model.encode(query).tolist()
    
    # Get sparse vector for query
    sparse_query = sparse_model.encode_queries(query)
    
    hdense, hsparse = hybrid_score_norm(dense_query, sparse_query, alpha)
    
    # Perform hybrid search
    results = index.query(
        vector=hdense,
        sparse_vector=hsparse,
        top_k=top_k,
        include_metadata=True
    )
    
    return results

In [4]:
# Test cell to verify hybrid search functionality

query = "How do I create a multi-language form?"

print("--- Dense-only search (alpha=1.0) ---")
dense_results = hybrid_search(query, top_k=3, alpha=1.0)
for match in dense_results['matches']:
    print(f"ID: {match['id']}, Score: {match['score']}")

print("\n--- Sparse-only search (alpha=0.0) ---")
sparse_results = hybrid_search(query, top_k=3, alpha=0.0)
for match in sparse_results['matches']:
    print(f"ID: {match['id']}, Score: {match['score']}")

print("\n--- Hybrid search (alpha=0.5) ---")
hybrid_results = hybrid_search(query, top_k=3, alpha=0.5)
for match in hybrid_results['matches']:
    print(f"ID: {match['id']}, Score: {match['score']}")

# Compare the results
dense_ids = [m['id'] for m in dense_results['matches']]
hybrid_ids = [m['id'] for m in hybrid_results['matches']]

if dense_ids == hybrid_ids:
    print("\n\nConclusion: 🚨 Hybrid search is NOT working. The dense and hybrid results are identical.")
else:
    print("\n\nConclusion: ✅ Hybrid search appears to be working. The results are different.")


--- Dense-only search (alpha=1.0) ---
ID: 0, Score: 0.771290064
ID: 1, Score: 0.712501943
ID: 11, Score: 0.687511802

--- Sparse-only search (alpha=0.0) ---
ID: 10, Score: 0.836942375
ID: 4, Score: 0.545208752
ID: 0, Score: 0.503409207

--- Hybrid search (alpha=0.5) ---
ID: 10, Score: 0.75395906
ID: 0, Score: 0.637349606
ID: 4, Score: 0.537609


Conclusion: ✅ Hybrid search appears to be working. The results are different.


In [4]:
query = "What languages can I use to create a form?"

hybrid_results = hybrid_search(query, top_k=3, alpha=0.5)

documents = [
    {"id": x["id"], "text": x["metadata"]["text"]} 
    for x in hybrid_results["matches"]
]

reranked_documents = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=query,
    documents=documents,
    top_n=3,
    return_documents=True,
    parameters={
        "truncate": "END"
    }
)

for doc in reranked_documents.data:
    print("index:", doc["index"])
    print("score:", doc["score"])
    print("text:", doc["document"]["text"])
    print("---\n")

index: 0
score: 0.98475236
text: Page Title: Create Multi Language Forms

Create multi-language forms

Save time and reach a wider audience by creating a single form with multiple language options. You'll have the option to write your own translations or have AI translate your form for you.  Depending on your language settings, you can decide if you want to automatically have your form translated for respondents or give them the option to translate the form. The form will be translated if the respondent's browser language matches one of the translations you've added. All responses will be displayed in one place within your Results panel. You can create multi-language forms with the following languages: Arabic Catalan Chinese (simplified) Chinese (traditional) Croatian Danish Dutch English Estonian Finnish French German (formal) German (informal) Greek Hebrew Hungarian Italian Japanese Korean Norwegian Polish Portuguese Russian Spanish Swedish Turkish Ukrainian To create multi-language 

In [ ]:
# Test cell to verify hybrid search functionality

query = "How do I create a multi-language form?"

print("--- Dense-only search (alpha=1.0) ---")
dense_results = hybrid_search(query, top_k=3, alpha=1.0)
for match in dense_results['matches']:
    print(f"ID: {match['id']}, Score: {match['score']}")

print("\n--- Sparse-only search (alpha=0.0) ---")
sparse_results = hybrid_search(query, top_k=3, alpha=0.0)
for match in sparse_results['matches']:
    print(f"ID: {match['id']}, Score: {match['score']}")

print("\n--- Hybrid search (alpha=0.5) ---")
hybrid_results = hybrid_search(query, top_k=3, alpha=0.5)
for match in hybrid_results['matches']:
    print(f"ID: {match['id']}, Score: {match['score']}")

# Compare the results
dense_ids = [m['id'] for m in dense_results['matches']]
hybrid_ids = [m['id'] for m in hybrid_results['matches']]

if dense_ids == hybrid_ids:
    print("\n\nConclusion: 🚨 Hybrid search is NOT working. The dense and hybrid results are identical.")
else:
    print("\n\nConclusion: ✅ Hybrid search appears to be working. The results are different.")


--- Dense-only search (alpha=1.0) ---
ID: 0, Score: 0.771290064
ID: 1, Score: 0.712501943
ID: 11, Score: 0.687511802

--- Sparse-only search (alpha=0.0) ---
ID: 10, Score: 0.836942375
ID: 4, Score: 0.545208752
ID: 0, Score: 0.503409207

--- Hybrid search (alpha=0.5) ---
ID: 10, Score: 0.75395906
ID: 0, Score: 0.637349606
ID: 4, Score: 0.537609


Conclusion: ✅ Hybrid search appears to be working. The results are different.


In [ ]:
# Test cell to verify hybrid search functionality

query = "How do I create a multi-language form?"

print("--- Dense-only search (alpha=1.0) ---")
dense_results = hybrid_search(query, top_k=3, alpha=1.0)
for match in dense_results['matches']:
    print(f"ID: {match['id']}, Score: {match['score']}")

print("\n--- Sparse-only search (alpha=0.0) ---")
sparse_results = hybrid_search(query, top_k=3, alpha=0.0)
for match in sparse_results['matches']:
    print(f"ID: {match['id']}, Score: {match['score']}")

print("\n--- Hybrid search (alpha=0.5) ---")
hybrid_results = hybrid_search(query, top_k=3, alpha=0.5)
for match in hybrid_results['matches']:
    print(f"ID: {match['id']}, Score: {match['score']}")

# Compare the results
dense_ids = [m['id'] for m in dense_results['matches']]
hybrid_ids = [m['id'] for m in hybrid_results['matches']]

if dense_ids == hybrid_ids:
    print("\n\nConclusion: 🚨 Hybrid search is NOT working. The dense and hybrid results are identical.")
else:
    print("\n\nConclusion: ✅ Hybrid search appears to be working. The results are different.")


--- Dense-only search (alpha=1.0) ---
ID: 0, Score: 0.771290064
ID: 1, Score: 0.712501943
ID: 11, Score: 0.687511802

--- Sparse-only search (alpha=0.0) ---
ID: 10, Score: 0.836942375
ID: 4, Score: 0.545208752
ID: 0, Score: 0.503409207

--- Hybrid search (alpha=0.5) ---
ID: 10, Score: 0.75395906
ID: 0, Score: 0.637349606
ID: 4, Score: 0.537609


Conclusion: ✅ Hybrid search appears to be working. The results are different.


In [ ]:
# Test cell to verify hybrid search functionality

query = "How do I create a multi-language form?"

print("--- Dense-only search (alpha=1.0) ---")
dense_results = hybrid_search(query, top_k=3, alpha=1.0)
for match in dense_results['matches']:
    print(f"ID: {match['id']}, Score: {match['score']}")

print("\n--- Sparse-only search (alpha=0.0) ---")
sparse_results = hybrid_search(query, top_k=3, alpha=0.0)
for match in sparse_results['matches']:
    print(f"ID: {match['id']}, Score: {match['score']}")

print("\n--- Hybrid search (alpha=0.5) ---")
hybrid_results = hybrid_search(query, top_k=3, alpha=0.5)
for match in hybrid_results['matches']:
    print(f"ID: {match['id']}, Score: {match['score']}")

# Compare the results
dense_ids = [m['id'] for m in dense_results['matches']]
hybrid_ids = [m['id'] for m in hybrid_results['matches']]

if dense_ids == hybrid_ids:
    print("\n\nConclusion: 🚨 Hybrid search is NOT working. The dense and hybrid results are identical.")
else:
    print("\n\nConclusion: ✅ Hybrid search appears to be working. The results are different.")


--- Dense-only search (alpha=1.0) ---
ID: 0, Score: 0.771290064
ID: 1, Score: 0.712501943
ID: 11, Score: 0.687511802

--- Sparse-only search (alpha=0.0) ---
ID: 10, Score: 0.836942375
ID: 4, Score: 0.545208752
ID: 0, Score: 0.503409207

--- Hybrid search (alpha=0.5) ---
ID: 10, Score: 0.75395906
ID: 0, Score: 0.637349606
ID: 4, Score: 0.537609


Conclusion: ✅ Hybrid search appears to be working. The results are different.
